# Github 工具包

`Github` 工具包包含一系列工具，可以让 LLM 代理与 Github 仓库进行交互。
该工具包是 [PyGitHub](https://github.com/PyGithub/PyGithub) 库的一个封装。

有关 GithubToolkit 所有功能和配置的详细文档，请参阅 [API 参考](https://python.langchain.com/api_reference/community/agent_toolkits/langchain_community.agent_toolkits.github.toolkit.GitHubToolkit.html)。

## 设置

总的来说，我们将执行以下步骤：

1. 安装 pygithub 库
2. 创建一个 Github 应用
3. 设置你的环境变量
4. 使用 `toolkit.get_tools()` 将工具传递给你的代理

要启用对单个工具的自动追踪，请设置您的 [LangSmith](https://docs.smith.langchain.com/) API 密钥：

In [ ]:
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")
# os.environ["LANGSMITH_TRACING"] = "true"

### 安装

#### 1. 安装依赖

该集成在 `langchain-community` 中实现。我们还需要 `pygithub` 依赖：

In [ ]:
%pip install --upgrade --quiet  pygithub langchain-community

#### 2. 创建一个 GitHub 应用

[请遵循此处的说明](https://docs.github.com/en/apps/creating-github-apps/registering-a-github-app/registering-a-github-app)来创建并注册一个 GitHub 应用。请确保您的应用具有以下[仓库权限：](https://docs.github.com/en/rest/overview/permissions-required-for-github-apps?apiVersion=2022-11-28)

*   提交状态 (只读)
*   内容 (读写)
*   议题 (读写)
*   元数据 (只读)
*   拉取请求 (读写)

应用注册后，您必须授予其访问您希望它操作的每个仓库的权限。请在 [github.com 此处](https://github.com/settings/installations)使用应用设置。

#### 3. 设置环境变量

在初始化您的代理之前，需要设置以下环境变量：

*   **GITHUB_APP_ID** - 一个六位数的数字，可以在您应用的常规设置中找到。
*   **GITHUB_APP_PRIVATE_KEY** - 您的应用的私钥 .pem 文件的路径，或该文件的完整文本字符串。
*   **GITHUB_REPOSITORY** - 您希望机器人操作的 GitHub 仓库名称。必须遵循 \{用户名\}/\{仓库名\} 的格式。*请确保该应用已首先被添加到此仓库中！*
*   可选：**GITHUB_BRANCH** - 机器人将进行提交的分支。默认为 `repo.default_branch`。
*   可选：**GITHUB_BASE_BRANCH** - 您仓库的基础分支，PR 将基于此分支创建。默认为 `repo.default_branch`。

In [ ]:
import getpass
import os

for env_var in [
    "GITHUB_APP_ID",
    "GITHUB_APP_PRIVATE_KEY",
    "GITHUB_REPOSITORY",
]:
    if not os.getenv(env_var):
        os.environ[env_var] = getpass.getpass()

## 实例化

现在我们可以实例化我们的工具包：

In [2]:
from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
from langchain_community.utilities.github import GitHubAPIWrapper

github = GitHubAPIWrapper()
toolkit = GitHubToolkit.from_github_api_wrapper(github)

## 工具

查看可用工具：

In [3]:
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name)

Get Issues
Get Issue
Comment on Issue
List open pull requests (PRs)
Get Pull Request
Overview of files included in PR
Create Pull Request
List Pull Requests' Files
Create File
Read File
Update File
Delete File
Overview of existing files in Main branch
Overview of files in current working branch
List branches in this repository
Set active branch
Create a new branch
Get files from a directory
Search issues and pull requests
Search code
Create review request


这些工具的用途如下：

下文将详细介绍每个步骤。

1.  **获取 Issue** - 从代码仓库中获取 Issue。

2.  **获取单个 Issue** - 获取某个特定 Issue 的详细信息。

3.  **评论 Issue** - 在某个特定 Issue 下发布评论。

4.  **创建 Pull Request** - 从机器人的工作分支向基础分支创建一个 Pull Request。

5.  **创建文件** - 在代码仓库中创建一个新文件。

6.  **读取文件** - 从代码仓库中读取一个文件。

7.  **更新文件** - 更新代码仓库中的一个文件。

8.  **删除文件** - 从代码仓库中删除一个文件。

## 包含发布工具

默认情况下，该工具包不包含发布相关的工具。你可以在初始化工具包时，通过设置 `include_release_tools=True` 来包含它们：

In [ ]:
toolkit = GitHubToolkit.from_github_api_wrapper(github, include_release_tools=True)

设置 `include_release_tools=True` 将包含以下工具：

*   **获取最新发布版本** - 从仓库中获取最新的发布版本。

*   **获取发布版本列表** - 从仓库中获取最新的 5 个发布版本。

*   **获取指定发布版本** - 通过标签名称（例如 `v1.0.0`）从仓库中获取特定的发布版本。

## 在智能体中使用

我们将需要一个 LLM 或聊天模型：

import ChatModelTabs from "@theme/ChatModelTabs";

<ChatModelTabs customVarName="llm" />

In [4]:
# | output: false
# | echo: false

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

使用工具子集初始化 agent：

In [6]:
from langgraph.prebuilt import create_react_agent

tools = [tool for tool in toolkit.get_tools() if tool.name == "Get Issue"]
assert len(tools) == 1
tools[0].name = "get_issue"

agent_executor = create_react_agent(llm, tools)

并向其发出查询：

In [7]:
example_query = "What is the title of issue 24888?"

events = agent_executor.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the title of issue 24888?
================================== Ai Message ==================================
Tool Calls:
  get_issue (call_iSYJVaM7uchfNHOMJoVPQsOi)
 Call ID: call_iSYJVaM7uchfNHOMJoVPQsOi
  Args:
    issue_number: 24888
================================= Tool Message =================================
Name: get_issue

{"number": 24888, "title": "Standardize KV-Store Docs", "body": "To make our KV-store integrations as easy to use as possible we need to make sure the docs for them are thorough and standardized. There are two parts to this: updating the KV-store docstrings and updating the actual integration docs.\r\n\r\nThis needs to be done for each KV-store integration, ideally with one PR per KV-store.\r\n\r\nRelated to broader issues #21983 and #22005.\r\n\r\n## Docstrings\r\nEach KV-store class docstring should have the sections shown in the [Appendix](#appendix) below. The sectio

## API 参考

关于 `GithubToolkit` 所有功能和配置的详细文档，请参阅 [API 参考](https://python.langchain.com/api_reference/community/agent_toolkits/langchain_community.agent_toolkits.github.toolkit.GitHubToolkit.html)。